# 00 · Run all notebooks on Google Colab

[Open in Colab](https://colab.research.google.com/github/yeolab/encode-rbp-case-study/blob/main/notebooks/workflow/00_colab_run_all.ipynb), then choose Runtime > Run all. Nothing needs to be uploaded: the first code cell copies the repository, with its small data, onto the Colab machine.

This notebook runs notebooks 01 to 07 one after another on this Colab machine and saves each finished notebook, with all its tables and plots, to `My Drive/CS3_workflow/` (`executed/`), together with the figure (`figures/workflow/`).

With `LITE = True` (the default, in the next cell) it uses the small data in `lite_data/`: the BAM files cut down to the reads these analyses use. That takes about an hour and fits the free Colab plan. Output goes to `colab_output_lite/`.

With `LITE = False` it downloads everything (about 50 GB) and reruns rMATS genome-wide, which takes 5 to 8 hours. The large files then live on this Colab machine, which is why all seven notebooks run here, in one session. Output goes to `colab_output/`. Keep the browser tab open, or use a paid Colab plan that keeps running in the background.

On Google Colab, run the next cell first in every session. It copies this repository onto the Colab machine and, with `SAVE_TO_DRIVE = True`, saves results and the figure to `My Drive/CS3_workflow` (Colab asks for permission to connect your Drive). `LITE = True` uses a small subset of the data that fits free Colab. On your own computer the cell does nothing.

In [ ]:
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess
    REPO = "https://github.com/yeolab/encode-rbp-case-study"
    CLONE = "/content/encode-rbp-case-study"
    LITE = True    # True: small subset of the data, fits free Colab (about 1 h). False: all the data (5-8 h, ~50 GB)
    SAVE_TO_DRIVE = True    # results and the figure in My Drive/CS3_workflow; False: on this Colab machine only
    if not os.path.isdir(CLONE):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, CLONE], check=True)
    WORKFLOW = CLONE + "/notebooks/workflow"
    OUTPUT = None
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT = "/content/drive/MyDrive/CS3_workflow"
    sys.path.insert(0, WORKFLOW)
    import colab_setup
    colab_setup.setup(WORKFLOW, lite=LITE, output_folder=OUTPUT, rmats=True)

Options: `REP1_ONLY = True` computes only replicate 1 of the splice maps (the one the figure shows), which saves about 40 minutes; it is the default in lite mode. Set `START` to a later notebook to skip the ones already done.

In [ ]:
import os
import subprocess
import time
from pathlib import Path

REP1_ONLY = os.environ.get("CS3_LITE") == "1"     # lite mode: replicate 1 only, to stay near one hour
START = "01"
NOTEBOOKS = ["01_gene_search", "02_experiment_detail", "03_download", "04_shrna_rnaseq",
             "05_differential_splicing", "06_dpsi_per_exon", "07_integrate"]

if REP1_ONLY:
    os.environ["REPS"] = "rep1"
executed = Path(os.environ.get("CASE_STUDY_ROOT", "../..")).resolve() / "executed"
executed.mkdir(parents=True, exist_ok=True)
for name in NOTEBOOKS:
    if name[:2] < START:
        continue
    started = time.time()
    print(time.strftime("%H:%M"), "running", name, flush=True)
    run = subprocess.run(["jupyter", "nbconvert", "--to", "notebook", "--execute", name + ".ipynb",
                          "--output-dir", str(executed), "--ExecutePreprocessor.timeout=-1",
                          "--ExecutePreprocessor.kernel_name=python3"], capture_output=True, text=True)
    if run.returncode != 0:
        print(run.stderr[-3000:])
        raise RuntimeError(name + " stopped with an error (above). Fix it, then set START = '" + name[:2] + "' and run this cell again.")
    print("      finished in {:.0f} min: {}".format((time.time() - started) / 60, executed / (name + ".ipynb")), flush=True)
print("Figure:", Path(os.environ.get("CASE_STUDY_ROOT", "../..")).resolve() / "figures" / "workflow" / "CS3_PICALM_QKI_workflow.png")

In [ ]:
from IPython.display import Image

Image(str(Path(os.environ.get("CASE_STUDY_ROOT", "../..")) / "figures" / "workflow" / "CS3_PICALM_QKI_workflow.png"), width=900)